# Análisis interno de ESM-2

## Actividades 1, 2 y 3

Trabajo de Segundo Corte - Ciencia de Datos  
Universidad de Pamplona - Ingeniería de Sistemas

Este notebook desarrolla las primeras tres actividades:

1. **Marco teórico:** Transformers, tipos de arquitectura, tokens, embeddings, self-attention, MLM y ESM-2.
2. **Ejecución de ESM-2:** carga del modelo `facebook/esm2_t6_8M_UR50D`, tokenización y extracción de salidas internas.
3. **Inspección del código fuente:** identificación de tokenizer, embeddings, self-attention, capas Transformer, feed-forward y MLM head.

> Nota: este trabajo no entrena el modelo, no hace fine-tuning y no construye un chatbot. El objetivo es observar y explicar qué ocurre dentro de un transformer científico preentrenado.


# Actividad 1: Marco teórico

## Transformer original

El Transformer fue propuesto por Vaswani et al. (2017) en *Attention Is All You Need*. Su idea principal es procesar secuencias usando mecanismos de atención en lugar de recurrencia. En vez de leer una secuencia paso a paso como una RNN, el Transformer calcula relaciones entre todos los tokens mediante self-attention.

## Familias de Transformers

Existen tres familias principales:

- **Encoder-only:** procesa toda la entrada de forma bidireccional. Se usa para comprensión, clasificación, embeddings y masked language modeling. Ejemplo: BERT y ESM-2.
- **Decoder-only:** usa atención causal; cada token solo mira hacia atrás. Se usa para generación autoregresiva. Ejemplo: GPT.
- **Encoder-decoder:** un encoder comprende la entrada y un decoder genera una salida. Se usa en traducción y resumen.

ESM-2 es encoder-only porque analiza secuencias completas de aminoácidos y produce representaciones contextualizadas, no texto generado paso a paso.

## Tokens, embeddings y hidden states

En ESM-2, una proteína se representa como una cadena de aminoácidos. Cada aminoácido funciona como un token. El tokenizer convierte estos tokens en identificadores numéricos (`input_ids`). Luego, el modelo transforma esos IDs en embeddings iniciales.

La diferencia clave es:

- **Embedding inicial:** vector numérico asociado a un token antes de pasar por las capas Transformer.
- **Hidden state:** representación contextualizada después de pasar por atención y capas Transformer.

## Self-attention

La self-attention calcula cuánto debe atender cada token a los demás tokens. Usa tres matrices conceptuales:

- **Q, Query:** lo que busca una posición.
- **K, Key:** cómo se presenta cada posición ante las demás.
- **V, Value:** la información que aporta cada posición.

La fórmula simplificada es:

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

## Multi-head attention

En vez de una sola atención, el modelo usa varias cabezas. Cada cabeza puede capturar relaciones distintas en la secuencia. En proteínas, esto permite que el modelo aprenda patrones estadísticos entre aminoácidos.

## Masked Language Modeling

Masked Language Modeling consiste en ocultar algunos tokens y pedirle al modelo que los prediga usando el contexto. BERT usa este enfoque para lenguaje natural. ESM-2 usa una idea equivalente aplicada a proteínas: se ocultan aminoácidos y el modelo predice cuáles son probables en esa posición.

## ESM-2

ESM-2, de Meta AI/FAIR, es un modelo de lenguaje de proteínas. Recibe secuencias de aminoácidos y produce representaciones útiles para análisis científico. No es un chatbot. Sus salidas principales son embeddings, hidden states, logits para MLM y, si la implementación lo permite, atenciones.

Referencias principales: Vaswani et al. (2017), Devlin et al. (2018), Radford et al. (2018), Lin et al. (2023).


# Preparación del entorno

Esta celda instala las librerías necesarias. En Google Colab se puede ejecutar directamente.


In [ ]:
%pip install -q torch transformers matplotlib scikit-learn pandas tabulate


# Importación de librerías


In [ ]:
import inspect
import textwrap
import torch
import pandas as pd

from transformers import AutoTokenizer, EsmModel
from transformers.models.esm import modeling_esm

print("PyTorch:", torch.__version__)


# Actividad 2: Ejecución de ESM-2 preentrenado

Modelo usado:

facebook/esm2_t6_8M_UR50D

Secuencias:

- Original: `MKTAYIAKQRQISFVKSHFSRQDILD`
- Mutada: `MKTAFIAKQRQISFVKSHFSRQDILD`
- Alterada: `DLIDQRSFHSSKVFSIQRQKAIYATKM`


In [ ]:
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

SEQUENCES = {
    "Original": "MKTAYIAKQRQISFVKSHFSRQDILD",
    "Mutada_Y_a_F": "MKTAFIAKQRQISFVKSHFSRQDILD",
    "Alterada": "DLIDQRSFHSSKVFSIQRQKAIYATKM",
}

print(SEQUENCES)


## Cargar tokenizer y modelo

Se intenta cargar el modelo con `attn_implementation="eager"` porque algunas implementaciones optimizadas no devuelven mapas de atención aunque se pida `output_attentions=True`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    model = EsmModel.from_pretrained(MODEL_NAME, attn_implementation="eager")
    print("Modelo cargado con attn_implementation='eager'.")
except TypeError:
    model = EsmModel.from_pretrained(MODEL_NAME)
    print("Modelo cargado sin parámetro attn_implementation.")

model.eval()

print("Tokenizer:", type(tokenizer))
print("Modelo:", type(model))


## Configuración del modelo

Aquí se reporta el número de capas, cabezas de atención, tamaño oculto y vocabulario.


In [ ]:
config = model.config

print("Modelo:", MODEL_NAME)
print("hidden_size:", config.hidden_size)
print("num_hidden_layers:", config.num_hidden_layers)
print("num_attention_heads:", config.num_attention_heads)
print("vocab_size:", config.vocab_size)
print("max_position_embeddings:", config.max_position_embeddings)


## Función para analizar una secuencia

La función:

1. Tokeniza la secuencia.
2. Ejecuta el modelo con `output_hidden_states=True` y `output_attentions=True`.
3. Extrae tokens, IDs, forma de `last_hidden_state` y forma de las atenciones.


In [ ]:
def analyze_sequence(name, sequence, tokenizer, model):
    inputs = tokenizer(sequence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    ids = inputs["input_ids"][0].tolist()

    attention_available = outputs.attentions is not None

    if attention_available:
        num_attention_tensors = len(outputs.attentions)
        attention_shape = tuple(outputs.attentions[0].shape)
    else:
        num_attention_tensors = 0
        attention_shape = None

    result = {
        "nombre": name,
        "secuencia": sequence,
        "longitud_sin_tokens_especiales": len(sequence),
        "tokens": tokens,
        "ids": ids,
        "input_ids_shape": tuple(inputs["input_ids"].shape),
        "last_hidden_state_shape": tuple(outputs.last_hidden_state.shape),
        "num_hidden_states": len(outputs.hidden_states) if outputs.hidden_states is not None else None,
        "attentions_disponibles": attention_available,
        "num_attention_tensors": num_attention_tensors,
        "attention_layer_0_shape": attention_shape,
        "outputs": outputs,
        "inputs": inputs,
    }

    return result


## Ejecutar ESM-2 sobre las tres secuencias


In [ ]:
results = {}
summary_rows = []

for name, sequence in SEQUENCES.items():
    result = analyze_sequence(name, sequence, tokenizer, model)
    results[name] = result

    print("\n" + "="*80)
    print("Secuencia:", name)
    print("Texto:", result["secuencia"])
    print("Longitud sin tokens especiales:", result["longitud_sin_tokens_especiales"])
    print("Tokens:", result["tokens"])
    print("IDs:", result["ids"])
    print("input_ids shape:", result["input_ids_shape"])
    print("last_hidden_state shape:", result["last_hidden_state_shape"])
    print("Número de hidden_states:", result["num_hidden_states"])
    print("¿Atenciones disponibles?:", result["attentions_disponibles"])
    print("Número de tensores de atención:", result["num_attention_tensors"])
    print("Forma atención capa 0:", result["attention_layer_0_shape"])

    summary_rows.append({
        "tipo": name,
        "longitud": result["longitud_sin_tokens_especiales"],
        "input_ids_shape": result["input_ids_shape"],
        "last_hidden_state_shape": result["last_hidden_state_shape"],
        "num_hidden_states": result["num_hidden_states"],
        "attentions_disponibles": result["attentions_disponibles"],
        "attention_layer_0_shape": result["attention_layer_0_shape"],
    })

df_summary = pd.DataFrame(summary_rows)
df_summary


## Interpretación inicial de las formas

En una secuencia de longitud `L`, el tokenizer puede agregar tokens especiales. Por eso la forma de `input_ids` suele ser algo como:

(batch_size, sequence_length_con_tokens_especiales)

El `last_hidden_state` suele tener forma:

(batch_size, sequence_length_con_tokens_especiales, hidden_size)

Para `facebook/esm2_t6_8M_UR50D`, el `hidden_size` normalmente es 320.

Si las atenciones están disponibles, una atención por capa suele tener forma:

(batch_size, num_heads, sequence_length, sequence_length)


# Actividad 3: Inspección del código fuente

En esta sección se inspeccionan las clases principales del modelo ESM dentro de Hugging Face Transformers.


## Arquitectura general del modelo

La instrucción `print(model)` muestra la estructura interna del modelo cargado.


In [ ]:
print(model)


## Tabla de componentes inspeccionados

| Componente | Clase esperada | Qué hace |
|---|---|---|
| Tokenizer | `EsmTokenizer` | Convierte aminoácidos y tokens especiales en IDs |
| Embeddings | `EsmEmbeddings` | Convierte IDs en vectores iniciales |
| Self-attention | `EsmSelfAttention` | Calcula Q, K, V y pesos de atención |
| Multi-head attention | Dentro de `EsmSelfAttention` | Procesa varias cabezas en paralelo |
| LayerNorm y residual | `EsmLayer`, `EsmOutput` | Estabiliza y preserva información |
| Feed-forward | `EsmIntermediate`, `EsmOutput` | MLP aplicada a cada posición |
| MLM head | `EsmLMHead` | Produce logits para tokens enmascarados |


## Función para mostrar fragmentos de código fuente

No se copian archivos completos. Solo se muestran fragmentos breves y anotables.


In [ ]:
def inspect_component(title, obj, max_lines=45):
    print("\n" + "="*80)
    print(title)
    print("="*80)

    try:
        file_path = inspect.getfile(obj)
        print("Archivo fuente:", file_path)
    except Exception as exc:
        print("No se pudo obtener archivo fuente:", exc)

    try:
        source = inspect.getsource(obj)
        lines = source.splitlines()
        fragment = "\n".join(lines[:max_lines])
        print("\nFragmento de código:")
        print(textwrap.indent(fragment, "    "))
        if len(lines) > max_lines:
            print(f"    ... fragmento recortado, total de líneas: {len(lines)}")
    except Exception as exc:
        print("No se pudo obtener código fuente:", exc)


## Inspección del tokenizer

El tokenizer convierte la secuencia de aminoácidos en tokens e IDs numéricos.


In [ ]:
inspect_component("Tokenizer usado por AutoTokenizer", type(tokenizer), max_lines=35)


## Inspección de embeddings

Los embeddings convierten `input_ids` en vectores iniciales. También se incorpora información posicional.


In [ ]:
inspect_component("Embeddings: EsmEmbeddings", modeling_esm.EsmEmbeddings, max_lines=45)


## Inspección de self-attention

Aquí se debe identificar dónde aparecen las proyecciones `query`, `key` y `value`, que corresponden a Q, K y V en la teoría.


In [ ]:
inspect_component("Self-Attention: EsmSelfAttention", modeling_esm.EsmSelfAttention, max_lines=70)


## Inspección de una capa Transformer

Una capa integra atención, feed-forward, normalización y conexiones residuales.


In [ ]:
inspect_component("Capa Transformer: EsmLayer", modeling_esm.EsmLayer, max_lines=55)


## Inspección de feed-forward

El bloque feed-forward transforma cada posición después de la atención.


In [ ]:
inspect_component("Feed-forward intermedio: EsmIntermediate", modeling_esm.EsmIntermediate, max_lines=40)
inspect_component("Salida feed-forward/residual: EsmOutput", modeling_esm.EsmOutput, max_lines=40)


## Inspección de MLM head

La cabeza MLM proyecta las representaciones al tamaño del vocabulario para predecir tokens ocultos.


In [ ]:
inspect_component("MLM Head: EsmLMHead", modeling_esm.EsmLMHead, max_lines=55)


# Conclusiones parciales de las Actividades 1, 2 y 3

1. ESM-2 es un Transformer científico aplicado a proteínas.
2. Es un modelo encoder-only, porque procesa la secuencia completa y genera representaciones contextualizadas.
3. Los tokens corresponden principalmente a aminoácidos y tokens especiales.
4. El `last_hidden_state` no es lo mismo que un embedding inicial: es una representación final contextualizada.
5. La atención usa Q, K y V para calcular relaciones entre posiciones.
6. El código fuente de Hugging Face permite ubicar componentes teóricos concretos como embeddings, self-attention, feed-forward y MLM head.


# Referencias

- Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.
- Radford, A. et al. (2018). *Improving Language Understanding by Generative Pre-Training*.
- Lin, Z. et al. (2023). *Evolutionary-scale prediction of atomic-level protein structure with a language model*. Science.
- Hugging Face Transformers Documentation: ESM models and model outputs.
